In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.blob import ContainerClient

TENANT_ID = "5a13decf-4ae8-423c-9d6f-320db1def1ad"
CLIENT_ID = "6fcdb218-bf19-483e-95ab-b86fb024790a"
secret = dbutils.secrets.get(catalog="workspace", schema="default", key="azure-reader-secret")

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, secret)
container = ContainerClient("https://stpowermarketfm.blob.core.windows.net", "raw", credential=credential)

In [0]:
names = [b.name for b in container.list_blobs(name_starts_with="prices/") if b.name.endswith(".json")]
print(len(names))
print(names[0])

80
prices/2026-10-05/prices_2026-10-05_NO1.json


In [0]:
import json
from datetime import datetime, timezone

loaded_at = datetime.now(timezone.utc).isoformat()
rows = []
for name in names:
    text = container.download_blob(name).readall()
    for r in json.loads(text):
        r["area"] = name[-8:-5]
        r["source_file"] = name
        r["loaded_at"] = loaded_at
        rows.append(r)

print(len(rows))

1920


In [0]:
bronze = spark.createDataFrame(rows)
bronze.write.mode("overwrite").saveAsTable("workspace.default.bronze_prices")
print(bronze.count())

1920
